- https://github.com/huggingface/transformers/issues/25296
- https://github.com/huggingface/transformers/pull/26176/files
- how do i make changes to a code base locally?

In [6]:
from transformers import (AutoModelForSequenceClassification, 
                          AutoConfig, 
                          BitsAndBytesConfig, 
                          BertForSequenceClassification)
from src.helper_fn import get_trainable_parameters

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='fp4', #'nf4'
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=False,
)

In [7]:
model_bert = BertForSequenceClassification.from_pretrained('bert-base-cased', 
                                                      num_labels=2, 
                                                      #quantization_config=bnb_config,
                                                      device_map=None)
get_trainable_parameters(model_bert)                                                      

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initi

'trainable params: 108311810 || all params: 108311810 || trainable%: 100.00'

In [12]:
model_name = 'roberta-base'
config = AutoConfig.from_pretrained(model_name, device_map='auto', quantization_config=bnb_config)
model = AutoModelForSequenceClassification.from_config(config)

In [13]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:

from accelerate import init_empty_weights
model_name = 'roberta-base'
config = AutoConfig.from_pretrained(model_name, device_map='auto', )
with init_empty_weights():
    model = AutoModelForSequenceClassification.from_config(config)
model.tie_weights()

In [19]:
import torch
# create a random tensor on device variable device
t = torch.tensor([[1,2,3],[4,5,6]])
t = t.to('mps')

In [ ]:
model(t).logits

In [5]:
# check model size
import sys
print('Size (MB):', sys.getsizeof(model_bert)/1e6)

Size (MB): 5.6e-05


In [15]:
from src.helper_fn import get_trainer
trainer, model = get_trainer(fine_tuning_name='lora', output_dir='temp')

2023-09-28 08:00:01,135 - src.logger - INFO - *****Getting Model and Tokenizer*****
Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification w

  0%|          | 0/2 [00:00<?, ?it/s]

2023-09-28 08:00:03,011 - src.logger - INFO - Capping dataset rows to 10000 train and 2000 test
Loading cached processed dataset at /Users/meninderpurewal/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61/cache-0681953129a5ca3a.arrow
Loading cached processed dataset at /Users/meninderpurewal/.cache/huggingface/datasets/yelp_polarity/plain_text/1.0.0/14f90415c754f47cf9087eadac25823a395fef4400c7903c5897f55cfaaa6f61/cache-3d3194378638209e.arrow
2023-09-28 08:00:03,210 - src.logger - INFO - Train shape: (10000, 2), Test shape: (2000, 2)
2023-09-28 08:00:03,211 - src.logger - INFO - Calculated eval_steps: 312
2023-09-28 08:00:03,212 - src.logger - INFO - *****Model input require grads enabled*****


In [20]:
trainer.model(t).logits

tensor([[ 0.5994, -0.3824],
        [ 0.7742, -0.3508]], device='mps:0', grad_fn=<LinearBackward0>)